# Chương 4 — Tiền xử lý ảnh và ablation study

Chương này đánh giá resize bằng SSIM/PSNR, không gian màu qua PCA, bốn chuẩn hóa, augmentation và t-SNE, PCA trên không gian đặc trưng, các bộ phát hiện cạnh và ablation study bằng Logistic Regression. Các thí nghiệm tốn tài nguyên dùng mẫu train có seed cố định; test split không tham gia fit transformer.

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
REPO_ROOT = next((p for p in _candidates if (p / "src").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Không tìm thấy src/. Hãy mở notebook trong repository.")
sys.path.insert(0, str(REPO_ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from src.data_loading import load_cifar10_dataset
from scipy.stats import ks_2samp, f_oneway
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from src.image_analysis import (
    compute_phashes, exact_duplicate_groups, near_duplicate_pairs,
    remove_exact_duplicate_indices, resize_and_reconstruct, convert_color_space,
    augment_image, edge_map, structural_similarity, peak_signal_noise_ratio,
)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(style="whitegrid", context="notebook")
print("Repository:", REPO_ROOT)

In [ ]:
X_train, y_train, X_test, y_test, class_names = load_cifar10_dataset()
# Recreate the same byte-identical-pixel filtering so this notebook is independently runnable.
hash_cache = REPO_ROOT / "data" / "processed" / "cifar10_train_phashes.npy"
if hash_cache.exists():
    phashes = np.load(hash_cache)
else:
    phashes = compute_phashes(X_train)
    hash_cache.parent.mkdir(parents=True, exist_ok=True)
    np.save(hash_cache, phashes)
exact_groups = exact_duplicate_groups(X_train)
exact_remove_indices = remove_exact_duplicate_indices(exact_groups)
keep_mask = np.ones(len(X_train), dtype=bool); keep_mask[exact_remove_indices] = False
X_train_clean, y_train_clean = X_train[keep_mask], y_train[keep_mask]
print("Train after byte-identical duplicate removal:", X_train_clean.shape)


## 1. Resize về ít nhất ba kích thước và so sánh SSIM/PSNR

Ảnh nguồn CIFAR-10 đã là 32×32. Thử 32×32, 64×64 và 128×128. Vì SSIM/PSNR yêu cầu ảnh cùng kích thước, ảnh sau resize được đưa trở lại 32×32 trước khi tính metric. Điều này đo mức mất mát do chuỗi resize→reconstruct; upscale không tạo thông tin chi tiết mới.

\[ PSNR = 10\log_{10}(MAX_I^2/MSE) \]

$$\mathrm{MSE}=\frac{1}{N}\sum_i (x_i-y_i)^2,\qquad \mathrm{PSNR}=10\log_{10}\frac{L^2}{\mathrm{MSE}}\ \text{(dB)},\quad L=255$$

$$\mathrm{SSIM}(x,y)=\frac{(2\mu_x\mu_y+c_1)(2\sigma_{xy}+c_2)}{(\mu_x^2+\mu_y^2+c_1)(\sigma_x^2+\sigma_y^2+c_2)},\quad c_1=(0.01L)^2,\ c_2=(0.03L)^2$$

SSIM được tính trên cửa sổ $7\times7$, lấy trung bình theo cửa sổ và theo kênh. Nếu $\mathrm{MSE}=0$ thì $\mathrm{PSNR}=\infty$.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
resize_n = min(300, len(X_train_clean))
resize_idx = rng.choice(len(X_train_clean), resize_n, replace=False)
resize_rows = []
for target_size in (32, 64, 128):
    ssim_values, psnr_values = [], []
    for idx in resize_idx:
        original = X_train_clean[idx]
        reconstructed = resize_and_reconstruct(original, target_size)
        ssim_values.append(structural_similarity(original, reconstructed, channel_axis=-1, data_range=255))
        psnr_values.append(peak_signal_noise_ratio(original, reconstructed, data_range=255))
    finite_psnr = [v for v in psnr_values if np.isfinite(v)]
    resize_rows.append({
        "target_size": f"{target_size}x{target_size}",
        "mean_SSIM": float(np.mean(ssim_values)), "std_SSIM": float(np.std(ssim_values)),
        "mean_PSNR_dB": float(np.mean(finite_psnr)) if finite_psnr else float("inf"),
        "std_PSNR_dB": float(np.std(finite_psnr)) if finite_psnr else 0.0,
    })
resize_results = pd.DataFrame(resize_rows)
display(resize_results.style.format({"mean_SSIM": "{:.4f}", "std_SSIM": "{:.4f}", "mean_PSNR_dB": "{:.2f}", "std_PSNR_dB": "{:.2f}"}))
plt.figure(figsize=(7, 4)); sns.lineplot(data=resize_results, x="target_size", y="mean_SSIM", marker="o")
plt.title("SSIM after resize and reconstruction"); plt.ylim(0, 1.02); plt.show()

**Phân tích:** SSIM càng gần 1 càng giống cấu trúc gốc; PSNR càng lớn thường biểu thị sai số bình phương càng nhỏ. So sánh cả độ lệch chuẩn. Vì ảnh CIFAR-10 gốc nhỏ, khi upscale rồi thu về kích thước cũ kết quả có thể gần giống ảnh ban đầu; khi triển khai mô hình ở kích thước lớn, cần xét chi phí tính toán và không được diễn giải upscale thành “tăng độ chi tiết”.

## 2. RGB, Grayscale, HSV, LAB — PCA k=50

Mỗi không gian màu được chuyển thành vector phẳng và PCA 50 thành phần (hoặc ít hơn nếu kích thước mẫu/feature không đủ). Explained variance là tổng tỷ lệ phương sai được giữ lại bởi 50 thành phần.

$$\mathrm{EVR}_k=\frac{\sum_{i=1}^{k}\lambda_i}{\sum_{i=1}^{d}\lambda_i}$$

$\lambda_i$ là trị riêng của ma trận hiệp phương sai (sắp giảm dần), $d$ là số chiều của vector đặc trưng, $k=50$ (hoặc ít hơn nếu mẫu không đủ).

In [ ]:
color_n = min(1_500, len(X_train_clean))
color_idx = rng.choice(len(X_train_clean), color_n, replace=False)
color_images = X_train_clean[color_idx]
color_rows = []
for space in ("RGB", "Grayscale", "HSV", "LAB"):
    converted = convert_color_space(color_images, space).astype(np.float32) / 255.0
    features = converted.reshape(len(converted), -1)
    k = min(50, features.shape[1], len(features) - 1)
    pca_color = PCA(n_components=k, svd_solver="randomized", random_state=RANDOM_SEED)
    pca_color.fit(features)
    color_rows.append({
        "color_space": space,
        "input_dimensions": features.shape[1],
        "PCA_components": k,
        "explained_variance_50": float(pca_color.explained_variance_ratio_.sum()),
    })
color_pca_results = pd.DataFrame(color_rows)
display(color_pca_results.style.format({"explained_variance_50": "{:.4f}"}))

**Phân tích:** tổng phương sai giải thích cao hơn không đồng nghĩa luôn phân loại tốt hơn; PCA là phép giảm chiều không giám sát. So sánh kết quả trên cùng mẫu ảnh và cùng k, sau đó dùng ablation ở phần sau để kiểm tra ảnh hưởng lên F1/accuracy.

## 3. Bốn phương pháp chuẩn hóa và kiểm định KS

So sánh Min-Max [0,1], Min-Max [-1,1], Z-score toàn tập và Z-score theo kênh. KS hai mẫu so sánh phân phối pixel trước/sau chuẩn hóa trên mẫu ngẫu nhiên. Do dữ liệu sau là phép biến đổi tất định của dữ liệu trước, p-value KS chỉ mô tả khác biệt phân phối trên thang đo; nó không phải kiểm định chất lượng ảnh hay chứng minh phân phối mới tốt hơn.

$$x'_{[0,1]}=\frac{x}{255},\qquad x'_{[-1,1]}=\frac{2x}{255}-1,\qquad x'_{z}=\frac{x-\mu}{\sigma}$$

Z-score toàn tập dùng một cặp $(\mu,\sigma)$ chung; Z-score theo kênh dùng $(\mu_c,\sigma_c)$ riêng cho từng kênh.

$$D_{n,m}=\sup_x\left|F_{1,n}(x)-F_{2,m}(x)\right|$$

$D_{n,m}$ là thống kê KS hai mẫu, $F_{1,n}$ và $F_{2,m}$ là hàm phân phối thực nghiệm trước và sau chuẩn hóa.

In [ ]:
norm_n = min(1_000, len(X_train_clean))
norm_idx = rng.choice(len(X_train_clean), norm_n, replace=False)
norm_images = X_train_clean[norm_idx].astype(np.float32) / 255.0
pixel_before = norm_images.reshape(-1, 3)
normalization_results = []

normalizations = {}
normalizations["MinMax [0,1]"] = norm_images.copy()
normalizations["MinMax [-1,1]"] = norm_images * 2.0 - 1.0
mu_all, sigma_all = norm_images.mean(), norm_images.std()
normalizations["Z-score global"] = (norm_images - mu_all) / max(sigma_all, 1e-8)
mu_channel = norm_images.mean(axis=(0, 1, 2), keepdims=True)
sigma_channel = norm_images.std(axis=(0, 1, 2), keepdims=True)
normalizations["Z-score per-channel"] = (norm_images - mu_channel) / np.maximum(sigma_channel, 1e-8)

for name, transformed in normalizations.items():
    pixel_after = transformed.reshape(-1, 3)
    for channel, channel_name in enumerate(["R", "G", "B"]):
        sample_before = pixel_before[:min(50_000, len(pixel_before)), channel]
        sample_after = pixel_after[:min(50_000, len(pixel_after)), channel]
        ks_stat, p_value = ks_2samp(sample_before, sample_after)
        normalization_results.append({"method": name, "channel": channel_name, "KS_statistic": ks_stat, "p_value": p_value})
normalization_results = pd.DataFrame(normalization_results)
display(normalization_results.style.format({"KS_statistic": "{:.4f}", "p_value": "{:.3g}"}))

## 4. Data augmentation — tối thiểu 5 phép biến đổi

Cài đặt 6 phép biến đổi: lật ngang, lật dọc, xoay 15°, random crop, Gaussian noise và điều chỉnh brightness/contrast. Augmentation chỉ áp dụng cho tập train; không augment test. Lật dọc có thể không phù hợp một số bài toán ảnh tự nhiên, vì vậy phải thảo luận tính hợp lý theo miền dữ liệu.

$$\text{Lật ngang: } x'(i,j)=x(i,\,W-1-j),\qquad \text{Nhiễu Gaussian: } x'=\operatorname{clip}(x+\varepsilon,0,255),\ \varepsilon\sim\mathcal N(0,\sigma^2)$$

$$\text{Độ sáng/tương phản: } x'=\operatorname{clip}(\alpha x+\beta,\,0,\,255)$$

Các tham số cụ thể (σ, α, β, góc xoay, kích thước crop) được cấu hình trong `augment_image`.

In [ ]:
augmentation_methods = [
    "horizontal_flip", "vertical_flip", "rotate_15",
    "random_crop", "gaussian_noise", "brightness_contrast"
]
example = X_train_clean[0]
fig, axes = plt.subplots(1, len(augmentation_methods) + 1, figsize=(15, 3))
axes[0].imshow(example); axes[0].set_title("Original"); axes[0].axis("off")
for ax, method in zip(axes[1:], augmentation_methods):
    augmented = augment_image(example, method, np.random.default_rng(RANDOM_SEED))
    ax.imshow(augmented); ax.set_title(method.replace("_", "\n"), fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# t-SNE: compare a deterministic set of original images and their horizontal flips.
tsne_n = min(250, len(X_train_clean))
tsne_idx = rng.choice(len(X_train_clean), tsne_n, replace=False)
original_batch = X_train_clean[tsne_idx]
augmented_batch = np.stack([
    augment_image(img, "horizontal_flip", np.random.default_rng(RANDOM_SEED + i))
    for i, img in enumerate(original_batch)
])
combined = np.concatenate([original_batch, augmented_batch]).astype(np.float32) / 255.0
combined_features = combined.reshape(len(combined), -1)
combined_pca = PCA(n_components=30, svd_solver="randomized", random_state=RANDOM_SEED).fit_transform(combined_features)
tsne_embedding = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", max_iter=500, random_state=RANDOM_SEED).fit_transform(combined_pca)
tsne_labels = np.concatenate([y_train_clean[tsne_idx], y_train_clean[tsne_idx]])
tsne_source = np.array(["Original"] * tsne_n + ["Augmented"] * tsne_n)
tsne_df = pd.DataFrame({"x": tsne_embedding[:, 0], "y": tsne_embedding[:, 1], "class": [class_names[i] for i in tsne_labels], "source": tsne_source})
plt.figure(figsize=(9, 7))
sns.scatterplot(data=tsne_df, x="x", y="y", hue="class", style="source", s=35, alpha=0.75)
plt.title("t-SNE: original vs horizontal-flip augmentation")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left"); plt.tight_layout(); plt.show()

**Phân tích:** augmentation hợp lý thường mở rộng phân phối đặc trưng mà không đổi nhãn. Với phép lật dọc hoặc xoay mạnh, ảnh có thể trở nên phi thực tế; quan sát t-SNE chỉ cho biết cấu trúc tương đối trong mẫu, không kết luận được mô hình tổng quát hóa tốt hơn nếu chưa có ablation.

## 5. [Nâng cao] PCA trên không gian đặc trưng ảnh

PCA được fit trên một mẫu train ngẫu nhiên tối đa 5.000 ảnh để phù hợp RAM/thời gian. Ghi rõ đây là ước lượng trên mẫu, không phải fit toàn bộ 50.000 ảnh. Scree plot cho từng PC; tính số thành phần tối thiểu để đạt 90%, 95%, 99% variance nếu nằm trong dải số PC đã tính.

$$\Sigma=\frac{1}{n-1}X_c^\top X_c,\qquad \Sigma v_i=\lambda_i v_i,\qquad Z=X_cV_k$$

Số thành phần $k_p$ là giá trị nhỏ nhất thỏa $\mathrm{EVR}_{k_p}\ge p$, với $p\in\{0.90,0.95,0.99\}$.

In [ ]:
pca_n = min(5_000, len(X_train_clean))
pca_idx = rng.choice(len(X_train_clean), pca_n, replace=False)
X_pca_raw = X_train_clean[pca_idx].astype(np.float32).reshape(pca_n, -1) / 255.0
pca_components = min(1_536, pca_n - 1, X_pca_raw.shape[1])
pca_full = PCA(n_components=pca_components, svd_solver="randomized", random_state=RANDOM_SEED)
X_pca = pca_full.fit_transform(X_pca_raw)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

scree_df = pd.DataFrame({"component": np.arange(1, pca_components + 1), "explained_variance": pca_full.explained_variance_ratio_, "cumulative_variance": cumulative_variance})
plt.figure(figsize=(10, 4)); plt.plot(scree_df["component"], scree_df["explained_variance"], marker=".")
plt.xlabel("Principal component"); plt.ylabel("Explained variance ratio"); plt.title("PCA scree plot"); plt.show()
variance_targets = []
for target in (0.90, 0.95, 0.99):
    position = np.searchsorted(cumulative_variance, target)
    count = int(position + 1) if position < len(cumulative_variance) else None
    variance_targets.append({"target_variance": target, "components_required_within_computed_range": count, "reached": count is not None})
display(pd.DataFrame(variance_targets))
print("Cumulative variance at computed component limit:", float(cumulative_variance[-1]))

In [ ]:
# 2D PCA visualization on a smaller balanced-like random subset.
pca_2d = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca_2d = pca_2d.fit_transform(X_pca_raw)
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y_train_clean[pca_idx], cmap="tab10", s=8, alpha=0.55)
plt.title("PCA 2D projection of CIFAR-10 training sample")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.colorbar(scatter, label="class id"); plt.show()

## 6. [Nâng cao] Edge detection: Sobel, Prewitt, Canny — 2 bộ tham số mỗi loại

- Sobel: đạo hàm ảnh theo trục x/y; so sánh kernel 3 và 5.
- Prewitt: gradient x/y bằng kernel Prewitt; so sánh kernel 3×3 và 5×5.
- Canny: hai cặp ngưỡng (50,150) và (100,200).

Edge Density = số pixel được đánh dấu cạnh / tổng số pixel.

$$G_x=\begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix}*I,\qquad G_y=\begin{bmatrix}-1&-2&-1\\0&0&0\\1&2&1\end{bmatrix}*I,\qquad |G|=\sqrt{G_x^2+G_y^2}$$

Prewitt dùng nhân tử toàn 1 thay cho hệ số 1-2-1. Pixel là cạnh khi $|G|$ vượt phân vị 85 (bộ A) hoặc 75 (bộ B). Canny dùng ngưỡng trễ (50, 150) cho bộ A và (100, 200) cho bộ B.

$$\mathrm{ED}=\frac{1}{HW}\sum_{i,j}E(i,j),\ E\in\{0,1\},\qquad F=\frac{\mathrm{SS}_B/(k-1)}{\mathrm{SS}_W/(N-k)}$$

$\mathrm{ED}$ là Edge Density của một ảnh; $F$ là thống kê ANOVA một chiều giữa $k$ lớp.

In [ ]:
edge_n = min(1_000, len(X_train_clean))
edge_idx = rng.choice(len(X_train_clean), edge_n, replace=False)
edge_rows = []
edge_visuals = {}
for method in ("Sobel", "Prewitt", "Canny"):
    for setting in ("A", "B"):
        densities = np.empty(edge_n, dtype=np.float32)
        for local_idx, original_idx in enumerate(edge_idx):
            edge = edge_map(X_train_clean[original_idx], method, setting)
            densities[local_idx] = float(edge.mean())
        edge_visuals[(method, setting)] = densities
        for class_id in np.unique(y_train_clean[edge_idx]):
            class_density = densities[y_train_clean[edge_idx] == class_id]
            edge_rows.append({"method": method, "setting": setting, "class_name": class_names[class_id], "mean_edge_density": class_density.mean(), "std_edge_density": class_density.std()})
edge_density_results = pd.DataFrame(edge_rows)
display(edge_density_results.head(30))

In [ ]:
# One-way ANOVA across class-specific edge-density distributions for each detector setting.
anova_rows = []
for (method, setting), densities in edge_visuals.items():
    groups = [densities[y_train_clean[edge_idx] == class_id] for class_id in np.unique(y_train_clean[edge_idx])]
    groups = [g for g in groups if len(g) > 1]
    if len(groups) >= 2:
        statistic, p_value = f_oneway(*groups)
        anova_rows.append({"method": method, "setting": setting, "F_statistic": statistic, "p_value": p_value})
edge_anova_results = pd.DataFrame(anova_rows)
display(edge_anova_results.style.format({"F_statistic": "{:.4f}", "p_value": "{:.3g}"}))

fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for ax, (method, setting) in zip(axes, [(m, s) for m in ("Sobel", "Prewitt", "Canny") for s in ("A", "B")]):
    ax.imshow(edge_map(X_train_clean[0], method, setting), cmap="gray")
    ax.set_title(f"{method} {setting}"); ax.axis("off")
plt.tight_layout(); plt.show()

## 7. Ablation study bằng Logistic Regression

So sánh một bộ phân loại đơn giản trước/sau các biểu diễn: RGB baseline, resize, các không gian màu, bốn chuẩn hóa, PCA-50, từng augmentation và hai bộ tham số của từng edge detector. Dùng cùng một train/test split phân tầng lấy mẫu; mọi transformer (PCA/scaler) chỉ fit trên train. Test luôn chứa ảnh gốc.

$$\mathrm{Acc}=\frac{1}{N}\sum_{i}\mathbf 1[\hat y_i=y_i],\qquad F1_c=\frac{2P_cR_c}{P_c+R_c},\qquad \text{macro-F1}=\frac{1}{C}\sum_{c=1}^{C}F1_c$$

$$P(y=c\mid x)=\frac{e^{w_c^\top x+b_c}}{\sum_{j}e^{w_j^\top x+b_j}}$$

Logistic Regression ước lượng xác suất lớp bằng softmax tuyến tính; macro-F1 lấy trung bình F1 của từng lớp nên không bị lớp đông lấn át.

In [ ]:
# Use a stratified, deterministic subset so the complete ablation is feasible on a 12-GB RAM laptop.
from sklearn.model_selection import train_test_split
subset_n = min(2_000, len(X_train_clean))
subset_idx = rng.choice(len(X_train_clean), subset_n, replace=False)
X_subset, y_subset = X_train_clean[subset_idx], y_train_clean[subset_idx]
X_ab_train, X_ab_test, y_ab_train, y_ab_test = train_test_split(
    X_subset, y_subset, test_size=0.25, stratify=y_subset, random_state=RANDOM_SEED
)
print("Ablation train/test:", X_ab_train.shape, X_ab_test.shape)

In [ ]:
# --- Ablation core: Logistic Regression, cùng một test split (ảnh gốc, chưa biến đổi) ---
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

ablation_rows = []


def evaluate_features(name, X_tr, X_te, y_tr, y_te):
    """Huấn luyện Logistic Regression trên X_tr và đánh giá trên X_te (ảnh gốc)."""
    clf = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)
    clf.fit(X_tr, y_tr)
    pred = clf.predict(X_te)
    return {
        "representation": name,
        "feature_dimensions": X_tr.shape[1],
        "accuracy": accuracy_score(y_te, pred),
        "macro_F1": f1_score(y_te, pred, average="macro"),
    }


def to_flat(images, scale=255.0):
    return images.reshape(len(images), -1).astype(np.float32) / scale


# 1) Baseline RGB: pixel đưa về [0,1] (trùng với Min-Max [0,1])
base_train = to_flat(X_ab_train)
base_test = to_flat(X_ab_test)
ablation_rows.append(evaluate_features("RGB baseline (Min-Max [0,1])", base_train, base_test, y_ab_train, y_ab_test))

# 2) Resize: thu/phóng rồi đưa về 32x32 để giữ cùng số chiều đầu vào
for size in (32, 64, 128):
    rs_train = np.stack([resize_and_reconstruct(img, size) for img in X_ab_train])
    rs_test = np.stack([resize_and_reconstruct(img, size) for img in X_ab_test])
    ablation_rows.append(evaluate_features(
        f"Resize {size}x{size} → 32x32", to_flat(rs_train), to_flat(rs_test), y_ab_train, y_ab_test))

# 3) Không gian màu: chuyển đổi rồi chuẩn hóa về [0,1]
for space in ("Grayscale", "HSV", "LAB"):
    ctr = convert_color_space(X_ab_train, space)
    cte = convert_color_space(X_ab_test, space)
    ablation_rows.append(evaluate_features(
        f"{space} color space", to_flat(ctr), to_flat(cte), y_ab_train, y_ab_test))

# 4) Chuẩn hóa: μ, σ chỉ tính trên train rồi áp dụng cho test
tr_f = X_ab_train.astype(np.float32) / 255.0
te_f = X_ab_test.astype(np.float32) / 255.0
mu_g, sd_g = tr_f.mean(), tr_f.std()
mu_c = tr_f.mean(axis=(0, 1, 2), keepdims=True)
sd_c = tr_f.std(axis=(0, 1, 2), keepdims=True)
norm_variants = {
    "Min-Max [-1,1]": lambda x: x * 2.0 - 1.0,
    "Z-score global": lambda x: (x - mu_g) / max(sd_g, 1e-8),
    "Z-score per-channel": lambda x: (x - mu_c) / np.maximum(sd_c, 1e-8),
}
for name, fn in norm_variants.items():
    ablation_rows.append(evaluate_features(
        name, fn(tr_f).reshape(len(tr_f), -1), fn(te_f).reshape(len(te_f), -1), y_ab_train, y_ab_test))

# 5) PCA-50: fit trên train, transform test
pca_ab = PCA(n_components=50, random_state=RANDOM_SEED).fit(base_train)
ablation_rows.append(evaluate_features(
    "PCA-50 (fit on train)", pca_ab.transform(base_train), pca_ab.transform(base_test), y_ab_train, y_ab_test))

# 6) Augmentation: chỉ thêm bản biến đổi vào TRAIN; test giữ ảnh gốc
aug_methods = ("horizontal_flip", "vertical_flip", "rotate_15", "random_crop", "gaussian_noise", "brightness_contrast")
aug_rng = np.random.default_rng(RANDOM_SEED)
for method in aug_methods:
    aug_imgs = np.stack([augment_image(img, method, rng=aug_rng) for img in X_ab_train])
    X_aug_train = np.concatenate([X_ab_train, aug_imgs])
    y_aug_train = np.concatenate([y_ab_train, y_ab_train])
    ablation_rows.append(evaluate_features(
        f"Augmentation: {method} (+train copies)", to_flat(X_aug_train), base_test, y_aug_train, y_ab_test))

print("Số cấu hình đã đánh giá trước edge detection:", len(ablation_rows))

In [ ]:
# Edge maps as local-feature representations; both parameter sets are included.
for edge_method in ("Sobel", "Prewitt", "Canny"):
    for edge_setting in ("A", "B"):
        edge_train = np.stack([edge_map(img, edge_method, edge_setting) for img in X_ab_train]).reshape(len(X_ab_train), -1).astype(np.float32)
        edge_test = np.stack([edge_map(img, edge_method, edge_setting) for img in X_ab_test]).reshape(len(X_ab_test), -1).astype(np.float32)
        ablation_rows.append(evaluate_features(f"{edge_method} edge map ({edge_setting})", edge_train, edge_test, y_ab_train, y_ab_test))

ablation_results = pd.DataFrame(ablation_rows).sort_values("macro_F1", ascending=False)
display(ablation_results.style.format({"accuracy": "{:.4f}", "macro_F1": "{:.4f}"}))
plt.figure(figsize=(11, 8)); sns.barplot(data=ablation_results, x="macro_F1", y="representation")
plt.title("Ablation comparison — macro F1 on untouched original test images")
plt.tight_layout(); plt.show()


### Đọc kết quả ablation

- Accuracy phản ánh tỷ lệ dự đoán đúng tổng thể; macro-F1 tính F1 từng lớp rồi lấy trung bình nên ít bị lớp đông lấn át.
- So sánh tất cả cấu hình trên cùng test split; chỉ thay đổi một nhóm kỹ thuật mỗi lần.
- Augmentation phải chỉ xuất hiện trong train. Nếu augmentation làm macro-F1 giảm, có thể phép biến đổi chưa phù hợp hoặc mô hình cơ sở chưa đủ khả năng tận dụng tính đa dạng.
- PCA-50 là nén thông tin tuyến tính; explained variance cao không bảo đảm giữ thông tin phân biệt lớp.

## Kết luận chương

Chốt kích thước ảnh theo SSIM/PSNR và chi phí tính toán; nêu không gian màu có explained variance cao và macro-F1 tốt; chọn chuẩn hóa có lý do; thảo luận t-SNE, PCA; ghi nhận edge detector có ANOVA p-value như thế nào; kết luận augmentation cải thiện hay không trên test chưa biến đổi. Không khái quát vượt quá mẫu thử đã sử dụng.